In [1]:
import gurobipy as gp
from gurobipy import GRB
import os
import pandas as pd
directorio=r"C:\Users\melco\OneDrive\Escritorio\pasantia_imple\datos_nuevos_mod"
d={}
B_num={}
valores = [
    970.45, 1642.3, 2543.4, 3272.3, 4006.8, 4776.3, 5610.6, 6137, 6785.6, 7363,
    8157.5, 8616.4, 9171.2, 9680.8, 10561, 11267, 12238, 12595, 13765, 14453,
    15444, 15563, 16398, 17379, 17733, 18357, 18794, 19730, 20478, 21634, 21994,
    22623, 23676, 24533, 25025, 25620, 26825, 27471, 28008, 28606, 29475, 30174,
    30392, 31150, 32213, 33283, 34133
]

# Crear un diccionario con índices desde 1
p = {i + 1: valores[i] for i in range(len(valores))}
print(p)
idx_p=gp.tuplelist(p)
maxi=0


for j, archivo in enumerate(os.listdir(directorio)):
    if archivo.endswith('.csv'):
        cont=0
        ruta_archivo = os.path.join(directorio, archivo)
        df = pd.read_csv(ruta_archivo)
        df['Distance'] = df['Distance'].fillna(0)
        df['dist_Acum'] = df['Distance'].cumsum()
        filtered_df = df[df['Speed[m/s]']==0]
        aux=df['dist_Acum'].iloc[-1]
        if aux>=maxi:
            maxi=aux
        k=0
        for i, row in filtered_df.iterrows():
            k=k+1
            d[(j+1, k)] = row['dist_Acum']
            cont=cont+1
            B_num[j+1]=cont
        #print(j)

B = {}

M=maxi
# Iterar sobre el diccionario original
for clave, valor in d.items():
    primer_valor = clave[0]
    if primer_valor not in B:
        B[primer_valor] = {}
    B[primer_valor][clave] = valor
    
B_max=max(list(B_num.values()))
print(B_max)

print(M)
idx_d=gp.tuplelist(d)
print(d)
#print(p)
#print(B[1])
print(B_num)
idx_B=gp.tuplelist(B_num)
print(idx_B)
#print(idx_d)
S=sum(B_num.values())
print(S)

{1: 970.45, 2: 1642.3, 3: 2543.4, 4: 3272.3, 5: 4006.8, 6: 4776.3, 7: 5610.6, 8: 6137, 9: 6785.6, 10: 7363, 11: 8157.5, 12: 8616.4, 13: 9171.2, 14: 9680.8, 15: 10561, 16: 11267, 17: 12238, 18: 12595, 19: 13765, 20: 14453, 21: 15444, 22: 15563, 23: 16398, 24: 17379, 25: 17733, 26: 18357, 27: 18794, 28: 19730, 29: 20478, 30: 21634, 31: 21994, 32: 22623, 33: 23676, 34: 24533, 35: 25025, 36: 25620, 37: 26825, 38: 27471, 39: 28008, 40: 28606, 41: 29475, 42: 30174, 43: 30392, 44: 31150, 45: 32213, 46: 33283, 47: 34133}
159
36443.281530054395
{(1, 1): 0.0, (1, 2): 655.539254712026, (1, 3): 655.539254712026, (1, 4): 655.539254712026, (1, 5): 746.6568862331186, (1, 6): 1267.8077595527354, (1, 7): 1427.6626123439992, (1, 8): 2253.8789473955603, (1, 9): 2327.4841787553432, (1, 10): 3069.8957715104643, (1, 11): 3218.573160199313, (1, 12): 3853.9151453255286, (1, 13): 4038.4435598837563, (1, 14): 4592.977806347796, (1, 15): 5453.813399392211, (1, 16): 5984.278792156344, (1, 17): 6011.409247795486, 

In [3]:
########### 1 FASE #############
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y", lb=-GRB.INFINITY)
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z",ub=300)
alpha=m.addVar(vtype=GRB.CONTINUOUS, name="alpha",ub=1)
m.setObjective(alpha, GRB.MAXIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=48*alpha for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 2000
m.Params.NoRelHeurTime = 600
m.optimize()
m.write("datos_nuevos.lp")

Set parameter Username
Academic license - for non-commercial use only - expires 2025-05-14
Set parameter TimeLimit to value 2000
Set parameter NoRelHeurTime to value 600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 451840 rows, 227369 columns and 2003368 nonzeros
Model fingerprint: 0x3315ef71
Variable types: 4777 continuous, 222592 integer (222592 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 7e+04]
Found heuristic solution: objective -0.0000000
Presolve removed 394180 rows and 198274 columns
Presolve time: 1.12s
Presolved: 57660 rows, 29095 columns, 189229 nonzeros
Variable types: 41 continuous, 29054 integer (29054 binary)
Starting NoRel heuristic
Found h

In [ ]:
############## 2 FASE #################
m = gp.Model()
x=m.addVars(idx_d,idx_p, vtype = GRB.BINARY, name="x")
y=m.addVars(idx_B, vtype=GRB.CONTINUOUS, name="y",lb=-GRB.INFINITY)
z=m.addVars(idx_d,vtype=GRB.CONTINUOUS, name="z", ub=300)
m.setObjective(z.sum("*","*"), GRB.MINIMIZE)#-x.sum("*","*","*")
S=sum(B_num.values())
m.addConstrs((x.sum(i,"*","*")>=48*0.91667 for i in idx_B),"max_base")
m.addConstrs((x.sum(i,j,"*")<=1 for (i,j) in idx_d ),"max_punto") #for i in idx_B for j in range(1,B_max+1)
m.addConstrs((z[i,j]>=d[i,j]+y[i]-p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_inf")
m.addConstrs((z[i,j]>=-d[i,j]-y[i]+p[k]-M*(1-x[i,j,k]) for (i,j) in idx_d for k in idx_p),"cota_sup")
#m.addConstr(x.sum("*","*","*")>=(0.1*S),"num_datos")
m.addConstrs((x.sum(i,"*",k)<=1 for i in idx_B for k in idx_p),"unicidad")
m.Params.TimeLimit = 19000
m.Params.NoRelHeurTime = 18000
m.optimize()

Set parameter TimeLimit to value 19000
Set parameter NoRelHeurTime to value 18000
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (win64 - Windows 11.0 (22631.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-1255U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 451840 rows, 227368 columns and 2003328 nonzeros
Model fingerprint: 0xe9b7c102
Variable types: 4776 continuous, 222592 integer (222592 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+04]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 3e+02]
  RHS range        [1e+00, 7e+04]
Presolve removed 390760 rows and 193538 columns
Presolve time: 1.21s
Presolved: 61080 rows, 33830 columns, 250277 nonzeros
Variable types: 4776 continuous, 29054 integer (29054 binary)
Starting NoRel heuristic
Found phase-1 solution: relaxation 387662
Found phase-1 solution: relaxation 364659
Found phase-1 solution: relaxation 330766
Found phase-1 

In [9]:
m.write("test.lp")
if m.SolCount > 0:
    # Recuperar los valores de las variables
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    for k in idx_p:
        #print('\nFlujos optimos para {}:'.format(k))
        for i,j in idx_d:
            if vx[i,j,k] > 0:
                print("Base de datos {} punto elegido {} para la parada {}.".format(i, j,k))
    for i in idx_B:
        if vy[i]!=0:
             print(f"Desplazamiento de {i} : {vy[i]}") 
    for i,j in idx_d:
        if vz[i,j]!=0:
            print(f"El error del punto {j} con valor {d[i,j]}en la base {i} es {vz[i,j]}")

Base de datos 1 punto elegido 11 para la parada 1.
Base de datos 2 punto elegido 8 para la parada 1.
Base de datos 4 punto elegido 5 para la parada 1.
Base de datos 5 punto elegido 6 para la parada 1.
Base de datos 6 punto elegido 3 para la parada 1.
Base de datos 7 punto elegido 8 para la parada 1.
Base de datos 8 punto elegido 3 para la parada 1.
Base de datos 9 punto elegido 4 para la parada 1.
Base de datos 10 punto elegido 6 para la parada 1.
Base de datos 11 punto elegido 9 para la parada 1.
Base de datos 12 punto elegido 5 para la parada 1.
Base de datos 13 punto elegido 7 para la parada 1.
Base de datos 14 punto elegido 7 para la parada 1.
Base de datos 15 punto elegido 10 para la parada 1.
Base de datos 16 punto elegido 2 para la parada 1.
Base de datos 17 punto elegido 4 para la parada 1.
Base de datos 18 punto elegido 1 para la parada 1.
Base de datos 19 punto elegido 5 para la parada 1.
Base de datos 20 punto elegido 4 para la parada 1.
Base de datos 21 punto elegido 9 para

In [13]:
if m.SolCount > 0:
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    
    # Primero, agrupemos por base de datos
    for i in idx_B:  # Para cada base de datos
        print(f"\nPara la base de datos {i}:")
        
        # Usamos un conjunto para evitar duplicados
        for k in idx_p:  # Para cada parada
            points_for_stop = set()  # Usamos un conjunto para almacenar los puntos únicos
            
            for j in [j for _,j in idx_d if (i,j) in idx_d]:
                if vx[i,j,k] > 0:
                    points_for_stop.add(j)
            
            if points_for_stop:  # Si hay puntos para esta parada
                print(f"  Parada {k}:")
                for point in sorted(points_for_stop):  # Ordenamos los puntos para una mejor presentación
                    print(f"    - Punto elegido: {point}")
        
        # Mostrar el desplazamiento de la base si existe
        if vy[i] != 0:
            print(f"  Desplazamiento de la base: {vy[i]}")
        
        # Mostrar los errores asociados a esta base
        errors = {j: vz[i,j] for _,j in idx_d if (i,j) in idx_d and vz[i,j] != 0}
        if errors:
            print("  Errores:")
            for j, error in sorted(errors.items()):
                print(f"    - Punto {j}: {error}")


Para la base de datos 1:
  Parada 1:
    - Punto elegido: 11
  Parada 2:
    - Punto elegido: 13
  Parada 3:
    - Punto elegido: 15
  Parada 4:
    - Punto elegido: 16
  Parada 5:
    - Punto elegido: 18
  Parada 6:
    - Punto elegido: 20
  Parada 7:
    - Punto elegido: 21
  Parada 8:
    - Punto elegido: 23
  Parada 9:
    - Punto elegido: 24
  Parada 10:
    - Punto elegido: 26
  Parada 11:
    - Punto elegido: 31
  Parada 12:
    - Punto elegido: 34
  Parada 13:
    - Punto elegido: 38
  Parada 14:
    - Punto elegido: 39
  Parada 15:
    - Punto elegido: 41
  Parada 16:
    - Punto elegido: 42
  Parada 17:
    - Punto elegido: 44
  Parada 18:
    - Punto elegido: 45
  Parada 19:
    - Punto elegido: 46
  Parada 20:
    - Punto elegido: 49
  Parada 21:
    - Punto elegido: 52
  Parada 22:
    - Punto elegido: 54
  Parada 23:
    - Punto elegido: 56
  Parada 24:
    - Punto elegido: 59
  Parada 25:
    - Punto elegido: 60
  Parada 26:
    - Punto elegido: 63
  Parada 27:
    - Pu

In [45]:
s=[]
dic={}
if m.SolCount > 0:
    vx = m.getAttr('x', x)
    vy = m.getAttr('x', y) 
    vz = m.getAttr('x', z)
    
    # Primero, agrupemos por base de datos
    for i in idx_B:  # Para cada base de datos
        s=[]
        print(f"\nPara la base de datos {i}:")
        
        # Usamos un conjunto para evitar duplicados
        for k in idx_p:  # Para cada parada
            points_for_stop = set()  # Usamos un conjunto para almacenar los puntos únicos
            
            for j in [j for _,j in idx_d if (i,j) in idx_d]:
                if vx[i,j,k] > 0:
                    points_for_stop.add(j)
            
            if points_for_stop:  # Si hay puntos para esta parada
                for point in sorted(points_for_stop):  # Ordenamos los puntos para una mejor presentación
                    s.append(point)
        
        # Mostrar el desplazamiento de la base si existe
        if vy[i] != 0:
            print(f"  Desplazamiento de la base: {vy[i]}")
        
        # Mostrar los errores asociados a esta base
        errors = {j: vz[i,j] for _,j in idx_d if (i,j) in idx_d and vz[i,j] != 0}
        if errors:
            print("  Errores:")
            for j, error in sorted(errors.items()):
                print(f"    - Punto {j}: {error}")
        dic.update({i:s})
print(dic)
def obt(d):
    diccionario={}
    for base, points in dic.items():
        results = []
        for point in points:
            results.append(d[base, point])
        diccionario.update({base:results})       
    return diccionario
obt(d)


Para la base de datos 1:
  Errores:
    - Punto 11: 89.3805510224629
    - Punto 13: 72.16882491158321
    - Punto 15: 70.02025850024074
    - Punto 16: 66.47366574512125
    - Punto 18: 60.317291930055944
    - Punto 20: 63.84363090778788
    - Punto 21: 27.630037863375037
    - Punto 23: 6.3781894600979285
    - Punto 24: 32.19669373535726
    - Punto 26: 20.573635193766677
    - Punto 31: 19.363632861517544
    - Punto 34: 11.824625313638535
    - Punto 38: 0.8438528875267366
    - Punto 39: 6.701330641604727
    - Punto 41: 32.46976015748078
    - Punto 42: 18.02944283533725
    - Punto 44: 21.221668250262155
    - Punto 45: 24.33438464446226
    - Punto 46: 14.760238546143228
    - Punto 49: 0.8780218675383367
    - Punto 52: 61.32705868373159
    - Punto 54: 4.864582440000959
    - Punto 56: 3.5406791567002074
    - Punto 59: 112.52246074132927
    - Punto 60: 113.73830772175643
    - Punto 63: 102.60523528696285
    - Punto 66: 46.2538382645871
    - Punto 68: 22.64220825920347

{1: [780.0534489775353,
  1461.059175088415,
  2360.8807414997577,
  3103.292334254879,
  3887.3117080699426,
  4626.374369092209,
  5487.209962136623,
  6044.805810539899,
  6661.349306264647,
  7243.707364806232,
  8076.019367138484,
  8518.266374686362,
  9089.491852887522,
  9593.328669358398,
  10518.82376015748,
  11634.96644283534,
  12270.377668250261,
  12630.242384644465,
  14297.369761453858,
  15064.008021867541,
  16097.38505868373,
  16514.810582439997,
  17647.2596791567,
  18254.99846074133,
  18625.20330772175,
  19431.42623528696,
  19848.96283826459,
  20798.279208259202,
  21520.318361111476,
  23355.554770527913,
  23714.862951106996,
  24355.650116344437,
  25463.590235307598,
  26309.043294826446,
  26919.017665967283,
  27478.904215335697,
  27903.21459199001,
  28750.09040007685,
  29156.83205690132,
  29943.9244164994,
  30506.32003442602,
  31347.38045461964,
  32086.50442068054,
  32669.87473920925,
  34372.5042399938],
 2: [865.1331251072176,
  1446.1021310